# Filter Nepali YouTube Comments for a Benchmark Dataset

This notebook reads a CSV that contains a `comment` column (plus any number of other columns) and produces a cleaned, **representative** benchmark dataset.

Design goal: keep the dataset representative of how Nepali speakers actually write online, rather than aggressively stripping it down.

**Keep**
- Nepali (Devanagari): `यो भिडियो धेरै राम्रो छ।`
- Romanized Nepali: `yo video dherai ramro cha`
- Mixed Nepali–English: `Video ramro cha but editing could be better.`
- Emojis that contribute meaning: `Yo ta lastai ramro 🔥`

**Remove**
- Comments with no meaningful content (`😂😂😂`, `👍👍`)
- Extremely short comments that can't be classified (`wow`, `nice`, `ok`)
- Spam (`Subscribe my channel`, `Visit my website`)
- Exact duplicates
- Comments that are only usernames, hashtags, or links

**Consider filtering** (flagged, not auto-dropped)
- Very long comments
- Comments entirely in a non-Nepali language

The biggest mistake would be filtering out Roman Nepali — a huge fraction of Nepali social media is written that way — so it is explicitly kept.

## 1. Configuration

Edit these to match your file. Everything downstream reads from here.

In [ ]:
from pathlib import Path

# --- Paths ---
INPUT_CSV = "comments.csv"            # path to your source CSV
OUTPUT_CSV = "comments_filtered.csv"  # cleaned dataset (kept rows only)
REVIEW_CSV = "comments_review.csv"    # rows flagged for manual review

# --- Column names ---
COMMENT_COL = "comment"               # the column to filter on
SOURCE_VALUE = "youtube"              # value written into the `source` metadata field

# --- Thresholds (tune for your taxonomy) ---
MIN_TOKENS = 2          # comments with fewer real word-tokens are dropped...
SHORT_WHITELIST = set() # ...unless the lowercased text is a valid label here, e.g. {"ok"}
MAX_WORDS = 80          # comments longer than this are FLAGGED (not dropped)
DROP_DUPLICATES = True  # drop exact duplicate texts (keeps first occurrence)
DROP_NON_NEPALI = False # if True, entirely-other-language comments are dropped instead of flagged

assert Path(INPUT_CSV).exists() or True  # set the path above before running cell 3

## 2. Detection helpers

Pure functions, no pandas — easy to unit-test and reuse. They classify script, count emojis, and decide keep / drop / flag.

In [ ]:
import re
import unicodedata

# Devanagari Unicode block (covers Nepali script).
DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")
LATIN_RE = re.compile(r"[A-Za-z]")
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#\w+")
WORD_RE = re.compile(r"[\w\u0900-\u097F]+", re.UNICODE)

# A compact spam lexicon. Extend as you see new patterns in your data.
SPAM_PATTERNS = [
    r"subscribe\s+(to\s+)?my\s+channel",
    r"visit\s+my\s+(website|channel|page)",
    r"check\s+out\s+my\s+channel",
    r"follow\s+me\s+on",
    r"sub\s*4\s*sub",
    r"promo\s*code",
    r"click\s+(the\s+)?link",
]
SPAM_RE = re.compile("|".join(SPAM_PATTERNS), re.IGNORECASE)

# Common Roman-Nepali function words. Their presence is a strong signal that a
# Latin-script comment is Nepali (or code-mixed) rather than plain English.
ROMAN_NEPALI_HINTS = {
    "cha", "chha", "xa", "ramro", "naramro", "ho", "hoina", "timi", "hami",
    "malai", "timro", "mero", "hamro", "dai", "bhai", "didi", "bhayo", "garyo",
    "garnu", "lastai", "dherai", "ekdam", "sabai", "yo", "tyo", "kasto", "kati",
    "raicha", "rahecha", "vaye", "bhaye", "huncha", "huncha", "chaina", "thiyo",
    "ani", "tara", "ra", "pani", "matra", "nai", "hai", "hola", "jasto", "video",
}


def count_emojis(text: str) -> int:
    """Count emoji / pictographic / symbol code points."""
    n = 0
    for ch in text:
        cp = ord(ch)
        if (
            0x1F300 <= cp <= 0x1FAFF   # symbols & pictographs, emoji, etc.
            or 0x2600 <= cp <= 0x27BF  # misc symbols + dingbats
            or 0x1F000 <= cp <= 0x1F0FF
            or cp in (0x2764, 0x2B50, 0x2728)
            or unicodedata.category(ch) == "So"  # other symbols
        ):
            n += 1
    return n


def strip_emojis(text: str) -> str:
    return "".join(
        ch for ch in text
        if not (
            0x1F000 <= ord(ch) <= 0x1FAFF
            or 0x2600 <= ord(ch) <= 0x27BF
            or unicodedata.category(ch) == "So"
        )
    )


def word_tokens(text: str):
    """Word tokens with URLs / mentions / hashtags removed."""
    cleaned = HASHTAG_RE.sub(" ", MENTION_RE.sub(" ", URL_RE.sub(" ", text)))
    return WORD_RE.findall(cleaned)


def classify_script(text: str) -> str:
    """Return one of: devanagari, roman_nepali, code_mixed, latin, other, empty."""
    has_dev = bool(DEVANAGARI_RE.search(text))
    has_lat = bool(LATIN_RE.search(text))
    toks = [t.lower() for t in word_tokens(text)]
    has_roman_hint = any(t in ROMAN_NEPALI_HINTS for t in toks)

    if has_dev and has_lat:
        return "code_mixed"
    if has_dev:
        return "devanagari"
    if has_lat:
        return "roman_nepali" if has_roman_hint else "latin"
    if strip_emojis(text).strip() == "":
        return "empty"
    return "other"


def classify_language(script: str) -> str:
    """Coarse language label derived from the script classification."""
    return {
        "devanagari": "nepali",
        "roman_nepali": "nepali",
        "code_mixed": "nepali_english",
        "latin": "english",
        "empty": "none",
        "other": "other",
    }.get(script, "other")

In [ ]:
def evaluate(text):
    """Classify one comment.

    Returns a dict with the decision ('keep' | 'drop' | 'flag'), a reason,
    and the metadata fields recommended for the benchmark.
    """
    raw = "" if text is None else str(text)
    text = raw.strip()
    script = classify_script(text)
    emoji_n = count_emojis(text)
    toks = word_tokens(text)
    n_words = len(toks)
    text_no_emoji = strip_emojis(text).strip()

    meta = {
        "script": script,
        "language": classify_language(script),
        "emoji_count": emoji_n,
        "word_count": n_words,
        "source": SOURCE_VALUE,
    }

    def out(decision, reason):
        return {"decision": decision, "reason": reason, **meta}

    # --- REMOVE rules ---
    if text == "":
        return out("drop", "empty")

    # Only emojis / punctuation — no words at all.
    if n_words == 0:
        return out("drop", "no_meaningful_content")

    # Only URLs, mentions, or hashtags (word_tokens already strips those, so if
    # nothing survived but the raw text had them, it was link/handle-only).
    stripped_meta = HASHTAG_RE.sub("", MENTION_RE.sub("", URL_RE.sub("", text))).strip()
    if stripped_meta == "" or strip_emojis(stripped_meta).strip() == "":
        return out("drop", "only_links_mentions_or_hashtags")

    # Spam.
    if SPAM_RE.search(text):
        return out("drop", "spam")

    # Too short to classify — unless whitelisted as a valid label.
    if n_words < MIN_TOKENS and text_no_emoji.lower() not in SHORT_WHITELIST:
        return out("drop", "too_short")

    # --- CONSIDER FILTERING rules (flag, don't auto-drop) ---
    if n_words > MAX_WORDS:
        return out("flag", "very_long")

    if script in ("latin", "other"):
        if DROP_NON_NEPALI:
            return out("drop", "non_nepali")
        return out("flag", "possibly_non_nepali")

    # --- KEEP ---
    return out("keep", "ok")


# quick sanity checks
for sample in [
    "यो भिडियो धेरै राम्रो छ।",
    "yo video dherai ramro cha",
    "Video ramro cha but editing could be better.",
    "Yo ta lastai ramro 🔥",
    "😂😂😂😂",
    "nice",
    "Subscribe my channel",
    "This is a really great and well-made video, thanks!",
]:
    r = evaluate(sample)
    print(f"{r['decision']:5} | {r['reason']:28} | {r['script']:12} | {sample}")

## 3. Load the CSV and apply the filter

In [ ]:
import pandas as pd

df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df):,} rows, columns: {list(df.columns)}")

if COMMENT_COL not in df.columns:
    raise KeyError(
        f"Column '{COMMENT_COL}' not found. Available: {list(df.columns)}. "
        "Update COMMENT_COL in the config cell."
    )

# Evaluate every comment; expand the result dict into new columns.
evaluated = df[COMMENT_COL].apply(evaluate).apply(pd.Series)
df = pd.concat([df, evaluated], axis=1)

# Mark exact duplicates (keep first). Done after evaluate so 'drop' rows don't
# consume the 'first occurrence' slot for a text that also has a kept copy.
if DROP_DUPLICATES:
    norm = df[COMMENT_COL].astype(str).str.strip().str.lower()
    is_dup = norm.duplicated(keep="first")
    df.loc[is_dup & (df["decision"] == "keep"), ["decision", "reason"]] = ["drop", "duplicate"]

df["decision"].value_counts()

## 4. Inspect what is being dropped / flagged

Always eyeball this before trusting the output — the thresholds and spam lexicon are heuristics.

In [ ]:
print("Breakdown by reason:")
print(df.groupby(["decision", "reason"]).size().to_string())

print("\nScript distribution among KEPT rows:")
print(df.loc[df["decision"] == "keep", "script"].value_counts().to_string())

print("\nSample of dropped comments:")
cols = [COMMENT_COL, "reason", "script"]
display(df.loc[df["decision"] == "drop", cols].head(20))

print("Sample of flagged (review) comments:")
display(df.loc[df["decision"] == "flag", cols].head(20))

## 5. Write outputs

- `OUTPUT_CSV` — the kept rows, with metadata columns, ready for labelling.
- `REVIEW_CSV` — flagged rows (very long / possibly non-Nepali) for a human to decide.

Flagged rows are **not** in the kept set by default. If a review confirms some are fine, append them manually.

In [ ]:
META_COLS = ["script", "language", "emoji_count", "word_count", "source"]
keep_cols = list(df.columns.drop(["decision", "reason"]))

kept = df[df["decision"] == "keep"][keep_cols].reset_index(drop=True)
flagged = df[df["decision"] == "flag"][keep_cols + []].reset_index(drop=True)

kept.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
df[df["decision"] == "flag"][[COMMENT_COL, "reason"] + META_COLS].to_csv(
    REVIEW_CSV, index=False, encoding="utf-8"
)

print(f"Kept    : {len(kept):,} rows -> {OUTPUT_CSV}")
print(f"Flagged : {(df['decision'] == 'flag').sum():,} rows -> {REVIEW_CSV}")
print(f"Dropped : {(df['decision'] == 'drop').sum():,} rows")